## 打入包并读取数据

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
batch_size = 256
transform = transforms.ToTensor()
train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)
test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True
)
train_iter = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)
test_iter = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

## 初始化模型参数

nn.Flatten()（扁平化层）作用：它替代了你之前手写的 X.reshape((-1, W.shape[0])) 动作。机制：当 [batch_size, 1, 28, 28] 的四维图片张量流经它时，它默认会保留第 0 维（batch_size），将其余三个维度 $1 \times 28 \times 28$ 强行连乘拉平成一个维度（$784$）。输出形状变为 [batch_size, 784]。

In [3]:
net = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 10)
)
def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)
net.apply(init_weights)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=10, bias=True)
)

## 要单独计算 Softmax 概率，而是把 Softmax 和 交叉熵（Log 运算）合并为一个算子整体计算。

In [4]:
loss = nn.CrossEntropyLoss(reduction='none')

## 优化算法

In [5]:
trainer = torch.optim.SGD(net.parameters(),lr=0.1)

## 训练

In [6]:
def accuracy(y_hat, y):
    preds = y_hat.argmax(dim=1)
    cmp = preds.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum())


def evaluate_accuracy(net, data_iter):
    net.eval()

    metric_correct = 0.0
    metric_total = 0.0

    with torch.no_grad():
        for X, y in data_iter:
            y_hat = net(X)

            metric_correct += accuracy(y_hat, y)
            metric_total += y.numel()

    return metric_correct / metric_total

In [7]:
def train_epoch_ch3(net, train_iter, loss, trainer):
    net.train()

    metric_loss = 0.0
    metric_correct = 0.0
    metric_total = 0.0

    for X, y in train_iter:
        y_hat = net(X)

        l = loss(y_hat, y)

        trainer.zero_grad()
        l.mean().backward()
        trainer.step()

        metric_loss += float(l.sum())
        metric_correct += accuracy(y_hat, y)
        metric_total += y.numel()

    train_loss = metric_loss / metric_total
    train_acc = metric_correct / metric_total

    return train_loss, train_acc

In [8]:
def train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer):
    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch_ch3(
            net,
            train_iter,
            loss,
            trainer
        )

        test_acc = evaluate_accuracy(
            net,
            test_iter
        )

        print(
            f"epoch {epoch + 1}, "
            f"loss {train_loss:.4f}, "
            f"train acc {train_acc:.4f}, "
            f"test acc {test_acc:.4f}"
        )

In [9]:
num_epochs = 10

train_ch3(
    net,
    train_iter,
    test_iter,
    loss,
    num_epochs,
    trainer
)

C:\Users\hp\AppData\Local\Temp\ipykernel_18916\3928910833.py:17: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  metric_loss += float(l.sum())


epoch 1, loss 0.7864, train acc 0.7490, test acc 0.7782
epoch 2, loss 0.5696, train acc 0.8143, test acc 0.8130
epoch 3, loss 0.5260, train acc 0.8256, test acc 0.8112
epoch 4, loss 0.5001, train acc 0.8329, test acc 0.8218
epoch 5, loss 0.4854, train acc 0.8374, test acc 0.8212
epoch 6, loss 0.4740, train acc 0.8401, test acc 0.8191
epoch 7, loss 0.4656, train acc 0.8421, test acc 0.8287
epoch 8, loss 0.4586, train acc 0.8451, test acc 0.8259
epoch 9, loss 0.4517, train acc 0.8464, test acc 0.8344
epoch 10, loss 0.4469, train acc 0.8478, test acc 0.8338
